# 04_01_Loading_Dimensions_to_SQL_Server

In [1]:
import os
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text

import infrabel_punctuality as ip

In [2]:
PATH_PROCESSED = Path("../data/processed/")

# Table of Contents  

- [1. LOADING STATION](#1-loading-dimensions)

- [1.1. Preparing SQL Server Connection](#11-preparing-sql-server-connection)

- [1.2. Loading DataFrames from `processed` Directory](#12-loading-dataframes-from-processed-directory)

- [1.3. Loading Dimensions to SQL Server](#13-loading-dimensions-to-sql-server)
    - [1.3.1. Loading Dim_Station to SQL](#131-loading-dim_station-to-sql-server)
    - [1.3.2. Loading Dim_Train_Service to SQL Server](#132-loading-dim_train_service-to-sql-server)


## 1. LOADING DIMENSIONS

**LOADING STRATEGY**  

**This notebook loads the prepared dimension DataFrames into the SQL Server data warehouse.**  

1) It loads **two dimension tables** into SQL Server : `Dim_Station` and `Dim_Train_Service`.  

2) To avoid implicit type conversions between the DataFrame dtypes and the SQL column types, and to ensure better control over column typing, **both tables have already been created using SQL DDL scripts**: *03_03_dim_station* and *03_04_dim_train_service* — located in `/sql/ddl/`.  

3) The loading strategy is a **full load**. Before each load run, a **TRUNCATE TABLE** statement is executed on the target tables to prevent duplicates while preserving the DDL-defined table structure defined.  

4) The loading process is performed through **SQLAlchemy** using the functions provided by the `sql_server_connection` module of the project's `infrabel_punctuality` package.  

5) The SQLAlchemy engine is configured to connect to the project's SQL Server database, `infrabel_punctuality_dwh`, created in *01_create_database* located in `/sql/ddl/`. The **database name** is therefore hardcoded in this notebook.  

6) The engine is also configured to connect to a local SQL Server instance using `localhost` as the default **server name**. If the SQL Server instance on your machine is not accessible through `localhost`, set the SQL_SERVER environment variable to the correct server or instance name.


## 1.1. Preparing SQL Server Connection

In this section:  

* The SQLAlchemy engine is created with the required parameters - driver, server and database names - and its connection to the database is tested.  

* The `select_sql_driver()` and `get_engine()` functions support both *ODBC Driver 17 for SQL Server* and *ODBC Driver 18 for SQL Server*.  

In [3]:
driver = ip.select_sql_driver()
server = os.getenv("SQL_SERVER", "localhost")
database = "infrabel_punctuality_dwh"

Driver: ODBC Driver 18 for SQL Server


In [4]:
engine = ip.get_engine(driver, server, database)

In [5]:
ip.test_connection(engine)

Connection successful to database: infrabel_punctuality_dwh


## 1.2. Loading DataFrames from `processed` Directory

In [6]:
Dim_Station = pd.read_parquet(PATH_PROCESSED / "Dim_Station.parquet")
Dim_Station.head()

,SK_Station,Ptcar_ID,Station_Name_French,Station_Name_Dutch,Station_Class,Station_Status,REFNIS_Municipality,Communes,Gemeenten,REFNIS_Province,Provinces,Provincies,REFNIS_Region,Regions,Gewesten,Population_Density_Category,lon,lat
0,1,443,GENK-ZUID-L.O.-FORD,GENK-ZUID-L.O.-FORD,Station,active,71016,Genk,Genk,70000,Province du Limbourg,Provincie Limburg,2000,Région flamande,Vlaams Gewest,towns_suburbs,5.497283,50.923363
1,2,275,CHARLEROI-A.T.,CHARLEROI-A.T.,Service installation,active,52011,Charleroi,Charleroi,50000,Province du Hainaut,Provincie Henegouwen,3000,Région wallonne,Waals Gewest,cities,4.458827,50.400574
2,3,1888,WONDELGEM-BUNDEL,WONDELGEM-BUNDEL,Station,active,44021,Gand,Gent,40000,Province de Flandre orientale,Provincie Oost-Vlaanderen,2000,Région flamande,Vlaams Gewest,cities,3.718856,51.085559
3,4,2155,ZEEBRUGGE-BRAM-TANKINST.,ZEEBRUGGE-BRAM-TANKINST.,Service installation,active,31005,Bruges,Brugge,30000,Province de Flandre occidentale,Provincie West-Vlaanderen,2000,Région flamande,Vlaams Gewest,towns_suburbs,3.24638,51.306214
4,5,2154,ZEEBRUGGE-BRAM-DSP880,ZEEBRUGGE-BRAM-DSP880,Station,active,31005,Bruges,Brugge,30000,Province de Flandre occidentale,Provincie West-Vlaanderen,2000,Région flamande,Vlaams Gewest,towns_suburbs,3.249265,51.304315


In [7]:
Dim_Train_Service = pd.read_parquet(PATH_PROCESSED / "Dim_Train_Service.parquet")
Dim_Train_Service.head()

,SK_Train_Service,Operator_Train_Service,Relation_Category,Relation,Relation_Direction,Direction_Is_Inferred,Is_Local,Is_National
0,1,SNCB/NMBS,CHARTER,CHARTER,CHARTER,1,0,0
1,2,THI-FACT,EURST,EURST,EURST: AMSTERDAM CENTRAAL -> LONDON-ST-PANCRAS...,0,0,0
2,3,THI-FACT,EURST,EURST,EURST: AMSTERDAM CENTRAAL -> MARNE-LA-VALLEE-C...,0,0,0
3,4,THI-FACT,EURST,EURST,EURST: AMSTERDAM CENTRAAL -> PARIS-NORD,0,0,0
4,5,THI-FACT,EURST,EURST,EURST: KOLN HBF -> PARIS-NORD,0,0,0


## 1.3. Loading Dimensions to SQL Server

### 1.3.1. Loading `Dim_Station` to SQL Server

In [8]:
ip.full_load_to_sql_server(
    engine, 
    dataframe=Dim_Station, 
    table_name="Dim_Station", 
    schema="dim"
    )

### 1.3.2. Loading `Dim_Train_Service` to SQL Server

In [9]:
ip.full_load_to_sql_server(
    engine, 
    dataframe=Dim_Train_Service, 
    table_name="Dim_Train_Service", 
    schema="dim"
    )